# Federated MedMNIST 3D - Orchestrator

This notebook launches a **real** federated learning experiment on distributed Flower infrastructure.

```
┌──────────────────────────────────────────┐
│  HOST_SuperLink  (SuperLink + Orchestrator) │
│  flower-superlink                        │
│    :9092  ← SuperNodes connect here      │
│    :9093  ← flwr run connects here       │
└──────────────┬───────────────────────────┘
               │ gRPC
   ┌───────────┴────────────┐
   │                        │
┌──┴──────────────┐  ┌──────┴──────────────┐
│ HOST_SuperNode  │  │ HOST_SuperNode       │
│ SuperNode 0     │  │ SuperNode 1          │
│ local port 9094 │  │ local port 9096      │
└─────────────────┘  └─────────────────────┘
```

## Before running

1. **HOST_SuperLink** - start the SuperLink: `bash start_superlink.sh`
2. **HOST_SuperNode terminal 1** - `bash start_supernode.sh <SUPERLINK_IP>`
3. **HOST_SuperNode terminal 2** - `bash start_supernode.sh <SUPERLINK_IP> 9096`
4. Set `SUPERLINK_IP` in the cell below and run all cells.

> ⚠️ SuperNodes must have `torch` and `medmnist` already installed,
> or be started with `--allow-runtime-dependency-installation`.

## 0 - Imports & configuration

In [8]:
import os
import re
import socket
import subprocess
import sys
import time
from pathlib import Path

try:
    import tomllib
except ImportError:
    import tomli as tomllib  # Python 3.10
import tomli_w

# ── User configuration ────────────────────────────────────────────────────────
SUPERLINK_IP           = "192.168.x.x"   # <-- replace with HOST_SuperLink IP
SUPERLINK_FLEET_PORT   = 9092             # SuperNodes connect here
SUPERLINK_CONTROL_PORT = 9093             # flwr run connects here

NUM_ROUNDS     = 3
MIN_CLIENTS    = 2
NUM_PARTITIONS = 2
LOCAL_EPOCHS   = 1
LEARNING_RATE  = 0.001

# ── Paths ─────────────────────────────────────────────────────────────────────
FL_APP_DIR = Path("fl_medmnist3d").resolve()
FLWR_BIN   = str(Path(sys.executable).parent / "flwr")

print(f"Python      : {sys.executable}")
print(f"flwr binary : {FLWR_BIN}")
print(f"App dir     : {FL_APP_DIR}")
assert Path(FLWR_BIN).exists(), f"flwr not found at {FLWR_BIN} - wrong kernel?"
assert (FL_APP_DIR / "pyproject.toml").exists(), "pyproject.toml not found"

Python      : /home/leno/miniconda3/bin/python
flwr binary : /home/leno/miniconda3/bin/flwr
App dir     : /home/leno/FL/flower/FL_dist_demo/real_fed/fl_medmnist3d


In [9]:
def run_command(cmd: list, cwd: Path | None = None) -> int:
    """Run a command and stream its stdout/stderr to the notebook output."""
    print("$ " + " ".join(cmd))
    env = {**os.environ, "PYTHONUNBUFFERED": "1"}
    with subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, cwd=str(cwd) if cwd else None, env=env,
    ) as proc:
        for line in proc.stdout:
            print(line, end="", flush=True)
    return proc.returncode


def check_port(host: str, port: int, timeout: float = 2.0) -> bool:
    """Return True if a TCP connection to host:port succeeds."""
    try:
        with socket.create_connection((host, port), timeout=timeout):
            return True
    except OSError:
        return False


def update_federation_address(address: str) -> None:
    """Write the SuperLink address into ~/.flwr/config.toml."""
    flwr_config = Path.home() / ".flwr" / "config.toml"
    flwr_config.parent.mkdir(parents=True, exist_ok=True)

    config = {}
    if flwr_config.exists():
        with open(flwr_config, "rb") as f:
            config = tomllib.load(f)

    config.setdefault("superlink", {}).setdefault("distributed", {})["address"] = address
    config["superlink"]["distributed"]["insecure"] = True

    with open(flwr_config, "wb") as f:
        tomli_w.dump(config, f)
    print(f"~/.flwr/config.toml updated: [superlink.distributed] address = {address}")

## 1 - Check SuperLink connectivity

In [10]:
fleet_ok   = check_port(SUPERLINK_IP, SUPERLINK_FLEET_PORT)
control_ok = check_port(SUPERLINK_IP, SUPERLINK_CONTROL_PORT)

print(f"Fleet API   {SUPERLINK_IP}:{SUPERLINK_FLEET_PORT}  → {'✓ reachable' if fleet_ok else '✗ NOT reachable'}")
print(f"Control API {SUPERLINK_IP}:{SUPERLINK_CONTROL_PORT}  → {'✓ reachable' if control_ok else '✗ NOT reachable'}")

if not (fleet_ok and control_ok):
    print("\n⚠  SuperLink is not reachable. Did you run `bash start_superlink.sh` on HOST_SuperLink?")
else:
    print("\n✓ SuperLink is up. Make sure SuperNodes on HOST_SuperNode are connected before the next cell.")

Fleet API   100.90.92.74:9092  → ✓ reachable
Control API 100.90.92.74:9093  → ✓ reachable

✓ SuperLink is up. Make sure SuperNodes on HOST_SuperNode are connected before the next cell.


## 2 - Launch distributed federation

`flwr run` packages the code into a FAB (Flower App Bundle),
sends it to the SuperLink, which automatically distributes it to the SuperNodes on HOST_SuperNode.

In [11]:
# Update federation config with the current SuperLink address
update_federation_address(f"{SUPERLINK_IP}:{SUPERLINK_CONTROL_PORT}")

print("\nSubmitting run to SuperLink...\n")
result = subprocess.run(
    [
        FLWR_BIN, "run", str(FL_APP_DIR), "distributed",
        "--run-config", f"num-server-rounds={NUM_ROUNDS}",
        "--run-config", f"min-clients={MIN_CLIENTS}",
        "--run-config", f"num-partitions={NUM_PARTITIONS}",
        "--run-config", f"local-epochs={LOCAL_EPOCHS}",
        "--run-config", f"learning-rate={LEARNING_RATE}",
    ],
    capture_output=True, text=True,
)
print(result.stdout)

if result.returncode != 0:
    print("✗ flwr run failed:")
    print(result.stderr)
else:
    match = re.search(r"run (\d+)", result.stdout)
    if not match:
        print("⚠ Could not extract run ID - check the output above.")
    else:
        run_id = match.group(1)
        print(f"Run ID: {run_id}\n")
        print("Streaming live logs...\n")
        rc = run_command([FLWR_BIN, "log", run_id, "distributed", "--stream"])
        if rc == 0:
            print("\n✓ Run completed successfully.")
        else:
            print(f"\n✗ Log stream ended with exit code {rc}.")

~/.flwr/config.toml updated: [superlink.distributed] address = 100.90.92.74:9093

Submitting run to SuperLink...

🎊 Successfully started run 15932587019740695406

Run ID: 15932587019740695406

Streaming live logs...

$ /home/leno/miniconda3/bin/flwr log 15932587019740695406 distributed --stream
INFO :      Starting logstream for run_id `15932587019740695406`
INFO :      Start `flwr-serverapp` process
INFO :      Starting Flower ServerApp, config: num_rounds=3, no round_timeout
INFO :      
INFO :      [INIT]
INFO :      Using initial global parameters provided by strategy
INFO :      Starting evaluation of initial global parameters
INFO :      Evaluation returned no results (`None`)
INFO :      
INFO :      [ROUND 1]
INFO :      configure_fit: strategy sampled 2 clients (out of 2)
INFO :      aggregate_fit: received 2 results and 0 failures
INFO :      configure_evaluate: strategy sampled 2 clients (out of 2)
INFO :      aggregate_evaluate: received 2 results and 0 failures
INFO :     

## 3 - Local test (optional)

Start SuperLink and 2 SuperNodes **on the same machine** to test without HOST_SuperNode.
Useful to verify the code works before switching to the real network.

In [12]:
procs = []
BIN_DIR       = Path(sys.executable).parent
SUPERLINK_BIN = str(BIN_DIR / "flower-superlink")
SUPERNODE_BIN = str(BIN_DIR / "flower-supernode")

def _start(cmd, label):
    p = subprocess.Popen(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    print(f"Started {label} (pid {p.pid})")
    procs.append(p)
    return p

def stop_all():
    for p in procs:
        try: p.terminate()
        except Exception: pass
    procs.clear()
    print("All background processes stopped.")

_start([SUPERLINK_BIN, "--insecure", "--database", ":flwr-in-memory:"], "SuperLink")
time.sleep(2)
_start([SUPERNODE_BIN, "--insecure", "--superlink", "127.0.0.1:9092",
        "--clientappio-api-address", "0.0.0.0:9094", "--max-retries", "0"], "SuperNode-0")
time.sleep(1)
_start([SUPERNODE_BIN, "--insecure", "--superlink", "127.0.0.1:9092",
        "--clientappio-api-address", "0.0.0.0:9096", "--max-retries", "0"], "SuperNode-1")
time.sleep(3)

update_federation_address("127.0.0.1:9093")
print("\nSubmitting local run...\n")
result = subprocess.run(
    [
        FLWR_BIN, "run", str(FL_APP_DIR), "distributed",
        "--run-config", f"num-server-rounds={NUM_ROUNDS}",
        "--run-config", "min-clients=2",
        "--run-config", "num-partitions=2",
        "--run-config", f"local-epochs={LOCAL_EPOCHS}",
        "--run-config", f"learning-rate={LEARNING_RATE}",
    ],
    capture_output=True, text=True,
)
print(result.stdout)

try:
    match = re.search(r"run (\d+)", result.stdout)
    if not match:
        print("⚠ Could not extract run ID.")
    else:
        run_id = match.group(1)
        print(f"Run ID: {run_id}\n")
        rc = run_command([FLWR_BIN, "log", run_id, "distributed", "--stream"])
        print(f"\n{'✓ Completed.' if rc == 0 else f'✗ Failed (exit {rc})'}")
finally:
    stop_all()

Started SuperLink (pid 75651)
Started SuperNode-0 (pid 75661)
Started SuperNode-1 (pid 75716)
~/.flwr/config.toml updated: [superlink.distributed] address = 127.0.0.1:9093

Submitting local run...

🎊 Successfully started run 14725858649883521579

Run ID: 14725858649883521579

$ /home/leno/miniconda3/bin/flwr log 14725858649883521579 distributed --stream
INFO :      Starting logstream for run_id `14725858649883521579`
INFO :      Start `flwr-serverapp` process
INFO :      Starting Flower ServerApp, config: num_rounds=3, no round_timeout
INFO :      
INFO :      [INIT]
INFO :      Using initial global parameters provided by strategy
INFO :      Starting evaluation of initial global parameters
INFO :      Evaluation returned no results (`None`)
INFO :      
INFO :      [ROUND 1]
INFO :      configure_fit: strategy sampled 4 clients (out of 4)
INFO :      aggregate_fit: received 4 results and 0 failures
INFO :      configure_evaluate: strategy sampled 4 clients (out of 4)
INFO :      aggre